In [30]:
import pandas as pd
import numpy as np
import re
from datetime import datetime, date
import os
# ─── Configuration ───────────────────────────────────────────────────────────
FILE2_PATH = "data/03_Library Systembook.csv"        # Customer ID, Customer Name
FILE1_PATH = "data/03_Library SystemCustomers.csv"        # Id, Books, Book checkout, Book Returned, Days allowed to borrow, Customer ID
OUTPUT_CLEAN = "output_clean.csv"
OUTPUT_DIRTY = "output_dirty.csv"
TODAY = date.today()

# ─── Summary counters ─────────────────────────────────────────────────────────
summary = {
    "file1_original_rows": 0,
    "file1_duplicates_removed": 0,
    "file1_nulls_removed": 0,
    "file1_clean_rows": 0,
    "file2_original_rows": 0,
    "file2_nulls_removed": 0,
    "file2_dirty_rows": 0,
    "file2_clean_rows": 0,
    "unmatched_customer_ids": 0,
}
dirty_records = []

# ─── Helper: parse flexible date strings ─────────────────────────────────────
def parse_date(val):
    """Try to parse a date value into a date object. Returns None on failure."""
    if pd.isna(val) or str(val).strip() == "":
        return None
    s = str(val).strip()
    formats = [
        "%d/%m/%Y", "%Y-%m-%d", "%m/%d/%Y", "%d-%m-%Y",
        "%d/%m/%y", "%Y/%m/%d", "%d %b %Y", "%d %B %Y",
    ]
    for fmt in formats:
        try:
            return datetime.strptime(s, fmt).date()
        except ValueError:
            pass
    # Handle Excel serial dates (numbers)
    try:
        n = float(s)
        if 20000 < n < 60000:
            return (datetime(1899, 12, 30) + pd.Timedelta(days=int(n))).date()
    except (ValueError, TypeError):
        pass
    return None


def format_date(d):
    """Format a date object as dd/mm/yyyy string."""
    return d.strftime("%d/%m/%Y") if d else ""


def extract_weeks(val):
    """Extract number of weeks from strings like '2 weeks', '14 days', '1 week'."""
    if pd.isna(val) or str(val).strip() == "":
        return None
    s = str(val).strip().lower()
    # Match patterns like "2 weeks", "1 week", "14 days"
    m = re.search(r"(\d+(?:\.\d+)?)\s*(week|weeks)", s)
    if m:
        return float(m.group(1))
    m = re.search(r"(\d+(?:\.\d+)?)\s*(day|days)", s)
    if m:
        return float(m.group(1)) / 7
    # Plain number → assume weeks
    m = re.match(r"^(\d+(?:\.\d+)?)$", s)
    if m:
        return float(m.group(1))
    return None


In [27]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1 — Clean File 1 (Customers)
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Cleaning File 1 (Customers) ──────────────────────────")
df1 = pd.read_csv(FILE1_PATH, dtype=str)
df1.columns = df1.columns.str.strip()


# Normalise column names flexibly
col_map1 = {}
for col in df1.columns:
    lc = col.lower().replace(" ", "").replace("_", "")
    if "customerid" in lc:
        col_map1[col] = "Customer ID"
    elif "customername" in lc or "name" in lc:
        col_map1[col] = "Customer Name"
df1.rename(columns=col_map1, inplace=True)

summary["file1_original_rows"] = len(df1)

# Remove rows where both columns are null
null_mask = df1["Customer ID"].isna() | (df1["Customer ID"].str.strip() == "") | \
            df1["Customer Name"].isna() | (df1["Customer Name"].str.strip() == "")
summary["file1_nulls_removed"] = null_mask.sum()
df1 = df1[~null_mask].copy()

# Strip whitespace
df1["Customer ID"] = df1["Customer ID"].str.strip()
df1["Customer Name"] = df1["Customer Name"].str.strip()

# Remove duplicates (keep first)
before = len(df1)
df1.drop_duplicates(subset=["Customer ID"], keep="first", inplace=True)
summary["file1_duplicates_removed"] = before - len(df1)
summary["file1_clean_rows"] = len(df1)

# Build lookup dict
customer_lookup = dict(zip(df1["Customer ID"], df1["Customer Name"]))
print(f"  Original rows       : {summary['file1_original_rows']}")
print(f"  Null rows removed   : {summary['file1_nulls_removed']}")
print(f"  Duplicates removed  : {summary['file1_duplicates_removed']}")
print(f"  Clean rows          : {summary['file1_clean_rows']}")




── Cleaning File 1 (Customers) ──────────────────────────
  Original rows       : 9
  Null rows removed   : 1
  Duplicates removed  : 0
  Clean rows          : 8


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2 — Clean File 2 (Checkouts)
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Cleaning File 2 (Checkouts) ──────────────────────────")
df2 = pd.read_csv(FILE2_PATH, dtype=str)
df2.columns = df2.columns.str.strip()
 
# Normalise column names
col_map2 = {}
for col in df2.columns:
    lc = col.lower().replace(" ", "").replace("_", "")
    if lc in ("id", "recordid", "checkoutid"):
        col_map2[col] = "Id"
    elif lc == "books" or lc == "book" or lc == "title":
        col_map2[col] = "Books"
    elif "checkout" in lc or "checkoutdate" in lc:
        col_map2[col] = "Book Checkout"
    elif "returned" in lc or "returndate" in lc:
        col_map2[col] = "Book Returned"
    elif "allowed" in lc or "dayallowed" in lc or "borrowperiod" in lc or "daysallowed" in lc:
        col_map2[col] = "Days Allowed to Borrow"
    elif "customerid" in lc:
        col_map2[col] = "Customer ID"
df2.rename(columns=col_map2, inplace=True)
 
summary["file2_original_rows"] = len(df2)
 
# Remove exact duplicate rows
before = len(df2)
df2.drop_duplicates(keep="first", inplace=True)
summary["file2_duplicates_removed"] = before - len(df2)
 
clean_rows = []
 
for idx, row in df2.iterrows():
    errors = []
    r = row.copy()
 
    # ── Early exit: Id and Books must be present ──────────────────────────────
    id_val = str(r.get("Id", "")).strip()
    books_val = str(r.get("Books", "")).strip()
 
    if not id_val or id_val.lower() == "nan":
        dirty_records.append({
            "Id": "", "Customer ID": r.get("Customer ID", ""), "Customer Name": "",
            "Books": books_val, "Book Checkout (raw)": r.get("Book Checkout", ""),
            "Book Returned (raw)": r.get("Book Returned", ""),
            "Days Allowed to Borrow": r.get("Days Allowed to Borrow", ""),
            "Errors": "Id is null/missing",
        })
        summary["file2_dirty_rows"] += 1
        continue
 
    if not books_val or books_val.lower() == "nan":
        dirty_records.append({
            "Id": id_val, "Customer ID": r.get("Customer ID", ""), "Customer Name": "",
            "Books": "", "Book Checkout (raw)": r.get("Book Checkout", ""),
            "Book Returned (raw)": r.get("Book Returned", ""),
            "Days Allowed to Borrow": r.get("Days Allowed to Borrow", ""),
            "Errors": "Books is null/missing",
        })
        summary["file2_dirty_rows"] += 1
        continue
 
    # ── Customer ID & Name ────────────────────────────────────────────────────
    cid = str(r.get("Customer ID", "")).strip()
    if not cid or cid.lower() == "nan":
        errors.append("Customer ID is null/missing")
        cid = ""
 
    customer_name = customer_lookup.get(cid, "")
    if cid and not customer_name:
        errors.append(f"Customer ID '{cid}' not found in customers file")
        summary["unmatched_customer_ids"] += 1
 
    # ── Book Checkout date ────────────────────────────────────────────────────
    checkout_raw = str(r.get("Book Checkout", "")).strip().strip('"').strip("'")
    checkout_date = parse_date(checkout_raw)
    checkout_fmt = ""
 
    if checkout_date is None:
        errors.append(f"Book Checkout date is invalid or missing (raw: '{checkout_raw}')")
    elif checkout_date > TODAY:
        errors.append(f"Book Checkout date '{format_date(checkout_date)}' is in the future")
        checkout_date = None
    else:
        checkout_fmt = format_date(checkout_date)
 
    # ── Book Returned date ────────────────────────────────────────────────────
    returned_raw = str(r.get("Book Returned", "")).strip().strip('"').strip("'")
    returned_date = parse_date(returned_raw)
    returned_fmt = ""
 
    if returned_raw and str(returned_raw).strip() and str(returned_raw).strip().lower() != "nan":
        if returned_date is None:
            errors.append(f"Book Returned date is invalid (raw: '{returned_raw}')")
        elif returned_date < TODAY:
            # Past return date is fine only if book was actually returned
            pass
        if returned_date and checkout_date and returned_date < checkout_date:
            errors.append(
                f"Book Returned '{format_date(returned_date)}' is before Book Checkout '{format_date(checkout_date)}'"
            )
            returned_date = None
        if returned_date:
            returned_fmt = format_date(returned_date)
 
    # ── Days Allowed to Borrow → Expected Return Date ─────────────────────────
    days_raw = r.get("Days Allowed to Borrow", "")
    weeks = extract_weeks(days_raw)
    expected_return_fmt = ""
 
    if not days_raw or str(days_raw).strip().lower() in ("", "nan"):
        errors.append("Days Allowed to Borrow is null/missing")
    elif weeks is None:
        errors.append(f"Days Allowed to Borrow value '{days_raw}' is unrecognisable")
    elif weeks <= 0:
        errors.append(f"Days Allowed to Borrow value '{days_raw}' is not positive")
    else:
        if checkout_date:
            expected_return = checkout_date + pd.Timedelta(weeks=weeks)
            expected_return_fmt = format_date(expected_return.date() if hasattr(expected_return, "date") else expected_return)
 
    # ── Route to clean or dirty ────────────────────────────────────────────────
    if errors:
        dirty_record = {
            "Id": r.get("Id", ""),
            "Customer ID": cid,
            "Customer Name": customer_name,
            "Books": books_val,
            "Book Checkout (raw)": checkout_raw,
            "Book Returned (raw)": returned_raw,
            "Days Allowed to Borrow": days_raw,
            "Errors": " | ".join(errors),
        }
        dirty_records.append(dirty_record)
        summary["file2_dirty_rows"] += 1
    else:
        clean_row = {
            "Id": r.get("Id", ""),
            "Customer ID": cid,
            "Customer Name": customer_name,
            "Books": books_val,
            "Book Checkout": checkout_fmt,
            "Book Returned": returned_fmt,
            "Days Allowed to Borrow": str(days_raw).strip(),
            "Book Expected Return Date": expected_return_fmt,
        }
        clean_rows.append(clean_row)
        summary["file2_clean_rows"] += 1
 


── Cleaning File 2 (Checkouts) ──────────────────────────
     Id                                     Books Book Checkout Book Returned  \
0     1                       Catcher in the Rye   "20/02/2023"    25/02/2023   
1     2          Lord of the rings the two towers  "24/03/2023"    21/03/2023   
2     3  Lord of the rings the return of the kind  "29/03/2023"    25/03/2023   
3     4                                The hobbit  "02/04/2023"    25/03/2023   
4     5                                     Dune   "02/04/2023"    25/03/2023   
5     6                              Little Women  "02/04/2023"    01/05/2023   
6     7                                        IT  "10/04/2063"    03/04/2023   
7     8                                   Misery   "15/04/2023"    03/04/2023   
8     9                                  Catch 22  "15/04/2023"    16/04/2023   
9    10                              Animal Farm   "20/04/2023"    24/04/2023   
10   11                                      1984 

In [29]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3 — Write outputs
# ═══════════════════════════════════════════════════════════════════════════════
df_clean = pd.DataFrame(clean_rows)
df_clean.to_csv(OUTPUT_CLEAN, index=False)

df_dirty = pd.DataFrame(dirty_records)
df_dirty.to_csv(OUTPUT_DIRTY, index=False)

print(f"  Original rows           : {summary['file2_original_rows']}")
print(f"  Duplicate rows removed  : {summary['file2_duplicates_removed']}")
print(f"  Dirty rows flagged      : {summary['file2_dirty_rows']}")
print(f"  Clean rows written      : {summary['file2_clean_rows']}")



PermissionError: [Errno 13] Permission denied: 'output_dirty.csv'

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4 — Summary report
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 60)
print("  PROCESSING SUMMARY")
print("═" * 60)
print(f"\n  FILE 1 — Customers")
print(f"    Original rows              : {summary['file1_original_rows']}")
print(f"    Null/empty rows removed    : {summary['file1_nulls_removed']}")
print(f"    Duplicate rows removed     : {summary['file1_duplicates_removed']}")
print(f"    Clean customer records     : {summary['file1_clean_rows']}")

print(f"\n  FILE 2 — Checkouts")
print(f"    Original rows              : {summary['file2_original_rows']}")
print(f"    Duplicate rows removed     : {summary['file2_duplicates_removed']}")
print(f"    Rows with issues (dirty)   : {summary['file2_dirty_rows']}")
print(f"    Unmatched Customer IDs     : {summary['unmatched_customer_ids']}")
print(f"    Clean rows written         : {summary['file2_clean_rows']}")

print(f"\n  OUTPUTS")
print(f"    Clean data  → {OUTPUT_CLEAN}  ({summary['file2_clean_rows']} rows)")
print(f"    Dirty data  → {OUTPUT_DIRTY}  ({summary['file2_dirty_rows']} rows)")

if dirty_records:
    print(f"\n  DIRTY RECORD BREAKDOWN")
    all_errors = []
    for d in dirty_records:
        all_errors.extend(d["Errors"].split(" | "))
    from collections import Counter
    for err_type, count in Counter(all_errors).most_common():
        print(f"    {count:>4}x  {err_type}")

print("\n" + "═" * 60)


════════════════════════════════════════════════════════════
  PROCESSING SUMMARY
════════════════════════════════════════════════════════════

  FILE 1 — Customers
    Original rows              : 9
    Null/empty rows removed    : 1
    Duplicate rows removed     : 0
    Clean customer records     : 8

  FILE 2 — Checkouts
    Original rows              : 114
    Duplicate rows removed     : 92
    Rows with issues (dirty)   : 22
    Unmatched Customer IDs     : 2
    Clean rows written         : 0

  OUTPUTS
    Clean data  → output_clean.csv  (0 rows)
    Dirty data  → output_dirty.csv  (22 rows)

  DIRTY RECORD BREAKDOWN
       4x  Book Checkout date is invalid or missing (raw: '"02/04/2023"')
       2x  Book Checkout date is invalid or missing (raw: '"15/04/2023"')
       2x  Book Checkout date is invalid or missing (raw: '"01/05/2023"')
       1x  Book Checkout date is invalid or missing (raw: '"20/02/2023"')
       1x  Book Checkout date is invalid or missing (raw: '"24/03/202